In [72]:
import os
import sys
import pandas as pd
import re
import unicodedata

In [73]:
# Ensure the scripts directory is on the import path
script_dir = r"c:\\Users\\ayo\\Desktop\\Grants_Alignment\\scripts"
if script_dir not in sys.path:
    sys.path.append(script_dir)


SOURCE_DIR = r"c:\\Users\\ayo\\Desktop\\Grants_Alignment\\media_data\\articles_by_year"
DEST_DIR = r"c:\\Users\\ayo\\Desktop\\Grants_Alignment\\media_data\\filtered_articles_by_year"

os.makedirs(DEST_DIR, exist_ok=True)


In [74]:


# First loop: gather all unique publishers for inspection
all_publishers = set()
for filename in os.listdir(SOURCE_DIR):
    if not filename.lower().endswith(".csv"):
        continue
    src_path = os.path.join(SOURCE_DIR, filename)
    df = pd.read_csv(src_path,encoding='utf-8')
    all_publishers.update(df["Publisher"].tolist())
print(f"Found {len(all_publishers)} unique publishers across all files.")

# Convert the set of all publishers to a DataFrame and write to CSV
publishers_df = pd.DataFrame({'Publisher': sorted(all_publishers)})
publishers_df.to_csv('C:\\Users\\ayo\\Desktop\\Grants_Alignment\\media_data\\all_publishers.csv', index=False)


Found 517 unique publishers across all files.


In [75]:




# ---------------------------------------------------------------------------
# 1. Publisher normalization
# ---------------------------------------------------------------------------
def normalize_publisher(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

# ---------------------------------------------------------------------------
# 2. Exclusion list (sports / entertainment / awards / trial coverage —
#    structurally can't produce agriculture journalism)
# ---------------------------------------------------------------------------
definitely_exclude = [
    "Radio Netherlands Worldwide", "African Women in Cinema (Accra)", "Ghana Football Association (Accra)",
    "African Wildlife Foundation (Washington, DC)", "Confederation of African Football (Egypt)", "Capital FM (Nairobi)",
    "South African Football Association (Johannesburg)", "Showmax (Johannesburg)", "Big Brother Africa (Johannesburg)",
    "African Elections Project (Accra)", "South African Sports Confederation and Olympic Committee (Johannesburg)",
    "Africa Top Sports (Lome)", "Gambia Radio & TV News (Banjul)", "South African Police Service (Pretoria)",
    "Big Brother Mzansi (Johannesburg)", "Afrotainment Museke Online Music Awards",
    # additions
    "Council of Southern African Football Associations (Johannesburg)",
    "Kickoff (Cape Town)",
    "Afropop Worldwide (New York)",
    "The Caine Prize for African Writing (London)",
    "Screen Africa (Johannesburg)",
    "Farafina Publishers (Lagos)",
    "Bemba Trial Website (The Hague)",
    "Wikileaks Cablegate (Sweden)",
]

# normalize the exclude list the same way publisher names are normalized,
# so casing/whitespace differences don't cause silent misses
exclude_set = {normalize_publisher(p) for p in definitely_exclude}




In [76]:
import re

stems = [
    r"\bagr[ioa]\w*",       # agronomy, agriculture, agroforestry (avoids "agree")
    r"\bfarm\w*",          # farm, farmer, farming, farmland
    r"\bcrop\w*",          # crop, crops, cropping, inter-cropping
    r"\bplant\w*",         # plant, plants, planting, plantation  <-- ADDED
    r"\birrig\w*",         # irrigation, irrigated
    r"\bfertil\w*",        # fertilizer, fertilizing, fertility
    r"\bharvest\w*",       # harvest, harvesting, post-harvest
    r"\bnutri\w*",         # nutrition, nutritional, nutrient, nutrients
    r"\bseed\w*",          # seed, seeds, seedling, seedbeds
    r"\bdrought\w*",       # drought, droughts
    r"\bfamin\w*",         # famine, famines
    r"\bstarv\w*",         # starvation, starving
    
    # Non-redundant additions
    r"\blandrace\w*",      # landrace, landraces (traditional local cultivars)
    r"\bgermplasm\w*",     # germplasm (seed genetic resources)
    r"\bresilien\w*",      # resilience, resilient (climate resilience)
    r"\bindigen\w*"        # indigenous (indigenous foods/plants)
]

phrases = [
    r"\bfood\s+(?:in)?security\b",
    r"\brural\s+development\b",
    r"\bunderutili[sz]ed\b",            # underutilized / underutilised
    r"\bNUS\b",                         # Neglected and Underutilized Species
    r"\bNUCS\b"                         # Neglected and Underutilized Crop Species
]

# Combine into a single, comprehensive regex
stem_pattern = "|".join(stems)
phrase_pattern = "|".join(phrases)
recall_pattern = f"{phrase_pattern}|{stem_pattern}"

ag_regex = re.compile(recall_pattern, flags=re.IGNORECASE)

In [77]:

# ---------------------------------------------------------------------------
# 4. Main loop — exclude BEFORE computing totals/counts/share
# ---------------------------------------------------------------------------
publisher_counts_by_file = {}

for filename in sorted(os.listdir(SOURCE_DIR)):
    if not filename.lower().endswith(".csv"):
        continue

    src_path = os.path.join(SOURCE_DIR, filename)
    df = pd.read_csv(src_path, encoding="utf-8")

    # normalize publisher names
    df["Publisher"] = df["Publisher"].apply(normalize_publisher)

    # drop excluded publishers FIRST — this is the key ordering fix
    mask_excluded = df["Publisher"].isin(exclude_set)
    filtered_df = df[~mask_excluded].copy()

    dest_path = os.path.join(DEST_DIR, filename)
    filtered_df.to_csv(dest_path, index=False)
    print(f"Filtered {filename}: {len(df)} -> {len(filtered_df)} rows "
          f"({mask_excluded.sum()} excluded)")

    # keyword match runs on the already-excluded dataframe
    keyword_mask = filtered_df["Body_Text"].str.contains(ag_regex, na=False)

    publisher_keyword_counts = (
        filtered_df.loc[keyword_mask]
        .groupby("Publisher")
        .size()
        .sort_values(ascending=False)
    )

    # totals computed on the same already-excluded dataframe,
    # so ag_share denominators are correct
    publisher_totals = filtered_df.groupby("Publisher").size()

    publisher_ag_share = (
        (publisher_keyword_counts / publisher_totals.reindex(publisher_keyword_counts.index))
        .sort_values(ascending=False)
    )

    publisher_counts_by_file[filename] = {
        "raw_counts": publisher_keyword_counts.to_dict(),
        "totals": publisher_totals.to_dict(),
        "ag_share": publisher_ag_share.round(3).to_dict(),
    }

    print(publisher_keyword_counts)

print("\nAggregated publisher keyword counts per file:\n", publisher_counts_by_file)

Filtered allafrica_2000.csv: 312 -> 311 rows (1 excluded)
Publisher
panafrican news agency (dakar)                                         33
un integrated regional information networks                            29
the post express (lagos)                                               23
the nation (nairobi)                                                   15
this day (lagos)                                                       14
vanguard (lagos)                                                       12
the news (lagos)                                                       10
united nations world food programme (rome)                              9
ghanaian chronicle (accra)                                              9
accra mail (accra)                                                      8
addis tribune (addis ababa)                                             7
new vision (kampala)                                                    7
concord times (freetown)                    

In [78]:
# After processing all files, you can inspect the aggregated dictionary
print("Aggregated publisher keyword counts per file:\n", publisher_counts_by_file)


Aggregated publisher keyword counts per file:
 {'allafrica_2000.csv': {'raw_counts': {'panafrican news agency (dakar)': 33, 'un integrated regional information networks': 29, 'the post express (lagos)': 23, 'the nation (nairobi)': 15, 'this day (lagos)': 14, 'vanguard (lagos)': 12, 'the news (lagos)': 10, 'united nations world food programme (rome)': 9, 'ghanaian chronicle (accra)': 9, 'accra mail (accra)': 8, 'addis tribune (addis ababa)': 7, 'new vision (kampala)': 7, 'concord times (freetown)': 6, 'p.m. news (lagos)': 5, 'the monitor (kampala)': 5, 'united states agency for international development (washington, dc)': 4, 'the east african (nairobi)': 4, 'all africa news agency': 3, 'tempo (lagos)': 3, 'business day (johannesburg)': 3, 'financial gazette (harare)': 3, 'newswatch (lagos)': 3, 'the perspective (smyrna, georgia)': 2, 'weekly trust (abuja)': 2, 'mail & guardian (johannesburg)': 2, 'food and agricultural organization (rome)': 2, 'action by churches together (geneva)': 2, 

In [79]:
# Convert the nested dictionary `publisher_counts_by_file` into a flat table
rows = []
for fname, data in publisher_counts_by_file.items():
    raw_counts   = data.get("raw_counts", {})
    totals       = data.get("totals", {})
    ag_shares    = data.get("ag_share", {})
    for pub in set(raw_counts) | set(totals):
        rows.append({
            "filename": fname,
            "publisher": pub,
            "raw_count": raw_counts.get(pub, 0),
            "total": totals.get(pub, 0),
            "ag_share": ag_shares.get(pub, 0.0),
        })

df_counts = pd.DataFrame(rows)
# Optional: sort by filename and publisher
df_counts.sort_values(["filename", "publisher"], inplace=True)

# Define the output path and write the dataframe to CSV
csv_out_path = os.path.join('C:\\Users\\ayo\\Desktop\\Grants_Alignment\\media_data',
                           'publisher_counts.csv')
df_counts.to_csv(csv_out_path, index=False)

print(f"Publisher counts written to {csv_out_path}")

Publisher counts written to C:\Users\ayo\Desktop\Grants_Alignment\media_data\publisher_counts.csv


In [80]:
df_counts.head()

,filename,publisher,raw_count,total,ag_share
19,allafrica_2000.csv,accra mail (accra),8,9,0.889
47,allafrica_2000.csv,action by churches together (geneva),2,3,0.667
37,allafrica_2000.csv,addis tribune (addis ababa),7,8,0.875
18,allafrica_2000.csv,african church information service (nairobi),1,1,1.000
48,allafrica_2000.csv,african eye news service (nelspruit),1,3,0.333


In [81]:
# Aggregate metrics across all years by publisher
agg = (
    df_counts
    .groupby('publisher')
    .agg(
        total_raw_count=('raw_count', 'sum'),
        total_articles=('total', 'sum')
    )
)

# Compute overall agriculture share per publisher
agg['overall_ag_share'] = agg['total_raw_count'] / agg['total_articles']

# Optional: mean share across years (if desired)
agg['mean_ag_share'] = df_counts.groupby('publisher')['ag_share'].mean()

# Reset index to have publisher as a column
agg = agg.reset_index()

# Sort by overall share descending
agg = agg.sort_values('total_raw_count', ascending=False)

# Display the aggregated table
agg.head(50)

,publisher,total_raw_count,total_articles,overall_ag_share,mean_ag_share
417,this day (lagos),2628,3521,0.746379,0.746760
464,vanguard (lagos),2079,3033,0.685460,0.682720
275,new vision (kampala),1929,2648,0.728474,0.745929
121,daily trust (abuja),1871,2547,0.734590,0.728792
389,the monitor (kampala),1594,2314,0.688850,0.694850
376,the herald (harare),1251,1638,0.763736,0.789870
396,the new times (kigali),1180,1584,0.744949,0.775850
391,the nation (nairobi),985,1982,0.496973,0.688850
241,leadership (abuja),867,1203,0.720698,0.707556
428,un integrated regional information networks,862,964,0.894191,0.901312


In [82]:
from collections import defaultdict

total_pubs = set()
matched_pubs = set()
total_article_counts = defaultdict(int)   # to see how much volume they had

for filename, d in publisher_counts_by_file.items():
    for pub, count in d["totals"].items():
        total_pubs.add(pub)
        total_article_counts[pub] += count
    for pub in d["raw_counts"].keys():
        matched_pubs.add(pub)

never_matched = sorted(
    total_pubs - matched_pubs,
    key=lambda p: total_article_counts[p],
    reverse=True
)

print(f"{len(never_matched)} publishers never matched an ag keyword across all 25 years\n")
for pub in never_matched:
    print(f"{pub}: {total_article_counts[pub]} total articles across all years")

48 publishers never matched an ag keyword across all 25 years

verdade (maputo): 4 total articles across all years
reporters sans frontiã ̈res (paris): 4 total articles across all years
aswat masriya (cairo): 3 total articles across all years
the gambia echo (raleigh): 3 total articles across all years
oxfam east africa blog (nairobi): 2 total articles across all years
all progressives congress (lagos): 2 total articles across all years
economic freedom fighters (johannesburg): 2 total articles across all years
itweb (johannesburg): 2 total articles across all years
united states congress (washington, dc): 2 total articles across all years
international coalition of civil society organizations: 2 total articles across all years
algerie presse service (algiers): 2 total articles across all years
mo ibrahim foundation (london): 1 total articles across all years
afrobarometer (accra): 1 total articles across all years
guernica (london): 1 total articles across all years
the independent (f

In [83]:
# Create sub‑folders for matched / unmatched articles
matched_dir = os.path.join(DEST_DIR, 'matched')
unmatched_dir = os.path.join(DEST_DIR, 'unmatched')
os.makedirs(matched_dir, exist_ok=True)
os.makedirs(unmatched_dir, exist_ok=True)

for fname in sorted(os.listdir(SOURCE_DIR)):
    if not fname.lower().endswith('.csv'):
        continue

    src_path = os.path.join(SOURCE_DIR, fname)
    df_year = pd.read_csv(src_path, encoding='utf-8')

    

    # Identify rows that match the agriculture regex
    mask = df_year['Body_Text'].str.contains(ag_regex, na=False)

    matched_df = df_year[mask]
    unmatched_df = df_year[~mask]

    # Write the split data to the corresponding folder
    matched_path = os.path.join(matched_dir, fname)
    unmatched_path = os.path.join(unmatched_dir, fname)

    matched_df.to_csv(matched_path, index=False)
    unmatched_df.to_csv(unmatched_path, index=False)

    print(f'{fname}: {len(matched_df)} matched, {len(unmatched_df)} unmatched')

allafrica_2000.csv: 249 matched, 63 unmatched
allafrica_2001.csv: 322 matched, 94 unmatched
allafrica_2002.csv: 437 matched, 118 unmatched
allafrica_2003.csv: 548 matched, 135 unmatched
allafrica_2004.csv: 952 matched, 255 unmatched
allafrica_2005.csv: 1278 matched, 1455 unmatched
allafrica_2006.csv: 1417 matched, 617 unmatched
allafrica_2007.csv: 1271 matched, 1035 unmatched
allafrica_2008.csv: 1681 matched, 715 unmatched
allafrica_2009.csv: 1743 matched, 500 unmatched
allafrica_2010.csv: 1502 matched, 504 unmatched
allafrica_2011.csv: 1974 matched, 636 unmatched
allafrica_2012.csv: 2230 matched, 591 unmatched
allafrica_2013.csv: 2259 matched, 702 unmatched
allafrica_2014.csv: 1996 matched, 1188 unmatched
allafrica_2015.csv: 2573 matched, 992 unmatched
allafrica_2016.csv: 1950 matched, 681 unmatched
allafrica_2017.csv: 1178 matched, 413 unmatched
allafrica_2018.csv: 892 matched, 275 unmatched
allafrica_2019.csv: 840 matched, 309 unmatched
allafrica_2020.csv: 1126 matched, 373 unmatche

In [84]:
# Total matched vs total unmatched articles across all years

# Sum of matched (raw) counts
matched_total = df_counts['raw_count'].sum()

# Sum of all articles (total counts)
total_articles = df_counts['total'].sum()

# Unmatched articles
unmatched_total = total_articles - matched_total

print(f"Total matched articles: {matched_total}")
print(f"Total unmatched articles: {unmatched_total}")
print(f"Overall match rate: {matched_total / total_articles:.4f}")

Total matched articles: 31000
Total unmatched articles: 12160
Overall match rate: 0.7183
